In [82]:
# загрузка данных

In [83]:
# обработка названий - убираем токены с цифрами, ловеркейсим, убираем тайтлы, обрезаем по числу слов

In [84]:
# лемматизируем названия в обоих трейнах и склеиваем обратно в названия

In [85]:
# лемматизируем названия категорий
# и как-то вручную надо будет для части из них расписать?

In [86]:
# если words_set категории вкладывается в word_set продукта, то назначаем эту категорию
#

In [100]:
from pymystem3 import Mystem

mystem = Mystem()
text = "коврик для мыши"
lemma = ''.join(mystem.lemmatize(text))
print(lemma)

коврик для мышь



In [88]:
import pandas as pd

In [89]:
cat_tree = pd.read_csv('category_tree.csv')
cat_tree.drop_duplicates('cat_name', inplace=True)

cat_tree['cat_name_words_bag'] = cat_tree['cat_name'].apply(lambda x:
                                                            set([w for w in mystem.lemmatize(x.lower()) if w.isalpha()]))
cat_tree.sample(5)

,cat_id,parent_id,cat_name,cat_name_words_bag
470,1539,148.0,Книги и учебники,"{книга, учебник, и}"
432,1463,142.0,Танцы и гимнастика,"{танец, гимнастика, и}"
1814,30804,12444.0,Сверла и наборы,"{сверло, набор, и}"
921,3542,462.0,Нумизматика и филателия,"{филателия, нумизматика, и}"
206,436,28.0,Игровые приставки,"{приставка, игровой}"


In [90]:
cat_tree.shape

(1709, 4)

In [91]:
train = pd.read_parquet('unlabeled_train.parquet')

In [92]:
from tqdm import tqdm

tqdm.pandas()

In [155]:
def lowercase(text):
    return text.lower()

def remove_words_with_digits(text):
    return ' '.join([w for w in text.split() if w.isalpha()])

def remove_english_upper_words(text):
    return ' '.join([w for w in text.split() if not(is_only_english_letters(w)) or not w.isupper()])

def truncate_name(text, words_count=6):
    return ' '.join([w for w in text.split()[:words_count]])

import re

def remove_product_title(text):
    return re.sub(r'\s*""[^""]+""', '', text).strip()

def is_only_english_letters(s):
    return bool(re.fullmatch(r"[A-Za-z]+", s))

In [156]:
from os import truncate
preprocess_functions = [remove_english_upper_words, remove_words_with_digits, remove_product_title, truncate_name]

def preprocess(text):
  for f in preprocess_functions:
    text = f(text)
  return text

In [157]:
train['clear_name'] = train['source_name'].apply(lambda x: preprocess(x))

In [158]:
mystem = Mystem()

train['name_words_bag'] = train['clear_name'].progress_apply(lambda x:
                                                            set(mystem.lemmatize(x.lower())))
train.sample()

100%|██████████| 784742/784742 [03:37<00:00, 3613.02it/s]


,source_name,name_words_bag,assigned_cat,clear_name
304816,Задняя крышка для Samsung M31 M315 черная,"{крышка, samsung, \n, для, , задний, черный}","[Крышки для посуды, Крышки для сковород, Крышк...",Задняя крышка для Samsung черная


In [160]:
train.sample(5)

,source_name,name_words_bag,assigned_cat,clear_name
122175,Процессор AMD Ryzen 9 5950X BOX,"{\n, , ryzen, процессор}",,Процессор Ryzen
525595,Электрогриль Redmond SteakMaster RGM-M816P,"{steakmaster, \n, , redmond, электрогриль}",,Электрогриль Redmond SteakMaster
287583,HairLab Машинка для стрижки многофункциональна...,"{стрижка, машинка, голубой, \n, hairlab, для, }",Машинки для стрижки,HairLab Машинка для стрижки голубой
778296,"Рюкзак для ноутбука 17,3-дюйма серый","{ноутбук, \n, серый, для, , рюкзак}","Запчасти для ноутбуков, Ноутбуки, Аккумуляторы...",Рюкзак для ноутбука серый
663329,Clifford Curzon: Decca Recordings 1949-1964 Vol.1,"{decca, clifford, \n, recordings, }",,Clifford Decca Recordings


In [161]:
def assign_category(name_words_bag, category_bag_list, cat_names):
  assigned = []
  for category_bag, category_name in zip(category_bag_list, cat_names):
    # if category_bag.issubset(name_words_bag):
    if len(category_bag.intersection(name_words_bag)) / len(category_bag) > 0.5:
      assigned.append(category_name)
  return assigned


train['assigned_cat'] = train['name_words_bag'].progress_apply(lambda x:
                                      assign_category(x, cat_tree['cat_name_words_bag'], cat_tree['cat_name']))

100%|██████████| 784742/784742 [11:44<00:00, 1113.13it/s]


In [162]:
train['assigned_cat'] = train['assigned_cat'].apply(lambda x: ', '.join(x))

In [175]:
train[train['assigned_cat'] != ''].sample(5)

,source_name,name_words_bag,assigned_cat,clear_name
602867,"Обруч Grace Dance профессиональный, дуга 18 мм...","{малиновый, dance, grace, \n, обруч, дуга, }",Обручи,Обруч Grace Dance дуга малиновый
77434,Подогреватель для бутылочек Maman RB-10,"{бутылочка, maman, подогреватель, \n, для, }","Подогреватели детских бутылочек, Соски для бут...",Подогреватель для бутылочек Maman
412198,Игровой компьютер StarsComp 2069007,"{\n, игровой, компьютер, , starscomp}",Игровые компьютеры,Игровой компьютер StarsComp
702439,Смартфон Xiaomi Redmi 10A_6934177776373 2/32 Г...,"{\n, серый, xiaomi, смартфон, redmi, }",Смартфоны,Смартфон Xiaomi Redmi серый
658825,"Коврик для мыши Cross Case, Cross PAD, черный","{мышь, коврик, \n, cross, для, , черный}","Коврики для мышек, Мыши",Коврик для мыши Cross Cross черный
